# Multi-sample cell typing

# Dependencies and preparation


In [ ]:
import anndata as ad

from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.spatial_plot import spatial_celltype_plot
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE
from scripts.utils import save_h5ad


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import scripts
import scripts.celltype_rules_IHOPE
import importlib

importlib.reload(scripts.celltype_rules_IHOPE)

Define sample base names:

In [ ]:
samples = [
    "IHOPE26_Spleen",

]

In [ ]:
for basename in samples:

    try:
        print(f"Processing {basename}")

        input_path = (
            f"../data/processed/anndata/"
            f"{basename}_filtered_arcsinh_cf5_GMM.h5ad"
        )

        output_path = (
            f"../data/processed/anndata/"
            f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"
        )

        # Load data
        adata = ad.read_h5ad(input_path)

        # Run cell typing
        adata = assign_cell_types_bool_IHOPE(adata)

        # Type plots
        celltype_cols = [
            c for c in adata.obs.columns
            if c.startswith("type_")
            and not c.endswith("unclassified")
            and not c.endswith("Endothelial")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols,
            min_cells=50
        )

        # T subtype plots
        t_subtypes = [
            c for c in adata.obs.columns
            if c.startswith("subtype_T")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols=t_subtypes,
            min_cells=15,
            size=10,
            alpha=0.7
        )

        # B subtype plots
        b_subtypes = [
            c for c in adata.obs.columns
            if c.startswith("subtype_B_")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols=b_subtypes,
            min_cells=15,
            size=10,
            alpha=0.7
        )

        # Summary table
        df_summary = summarize_celltypes_IHOPE(
            adata,
            filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary.csv"
        )

        # Save anndata
        save_h5ad(adata, output_path)

        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error in {basename}: {e}")
